# 2. Model Training
## MVTec Metal Nut - Binary Classification

Load prepared data, train ResNet18 with frozen backbone, save trained model

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np
import time
import os
import pickle
import json
from pathlib import Path

# For reproducibility
torch.manual_seed(43)
np.random.seed(43)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [5]:
#HYPERPARAMETERS
lr = 1e-3
momentum = 0.9
weight_decay = 1e-4
epochs = 20
optimizer_type = 'SGD'

## Load Data & Recreate DataLoaders

In [7]:
# Import DataLoader
from torch.utils.data import DataLoader, Subset, Dataset
from PIL import Image

# Define MVTecBinaryDataset class (needed for unpickling)
class MVTecBinaryDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.transform = transform
        self.images = []
        self.labels = []
        self.defect_types = ['bent', 'color', 'flip', 'scratch']
        self._load_dataset()
    
    def _load_dataset(self):
        split_dir = self.root_dir / self.split
        good_dir = split_dir / 'good'
        if good_dir.exists():
            for img_path in sorted(good_dir.glob('*.png')):
                self.images.append(str(img_path))
                self.labels.append(0)
        for defect_type in self.defect_types:
            defect_dir = split_dir / defect_type
            if defect_dir.exists():
                for img_path in sorted(defect_dir.glob('*.png')):
                    self.images.append(str(img_path))
                    self.labels.append(1)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Load datasets from notebook 1
datasets = pickle.load(open('../data/processed/datasets.pkl', 'rb'))
train_dataset = datasets['train_dataset']
test_dataset = datasets['test_dataset']
train_indices = datasets['train_indices']
val_indices = datasets['val_indices']

# Load config
config = pickle.load(open('../data/processed/config.pkl', 'rb'))
imagenet_mean = config['imagenet_mean']
imagenet_std = config['imagenet_std']
batch_size = config['batch_size']

# Create subsets
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

# Recreate dataloaders
num_workers = 0
pin_memory = True if torch.cuda.is_available() else False

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, 
                          num_workers=num_workers, pin_memory=pin_memory)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, 
                        num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                         num_workers=num_workers, pin_memory=pin_memory)

print("=== Loaded Data ===")
print(f"Train batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")
print(f"Num workers:        {num_workers}")

# Load dataset info for reference
with open('../data/processed/dataset_stats.json', 'r') as f:
    dataset_stats = json.load(f)

=== Loaded Data ===
Train batches:      6
Validation batches: 2
Test batches:       4
Num workers:        0


## Load ResNet18 and Modify for Binary Classification

In [8]:
# Load pretrained ResNet18
try:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
except Exception:
    model = models.resnet18(pretrained=True)

# Replace final layer for binary classification
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print("=== Model Loaded ===")
print(f"Architecture: ResNet18 (pretrained on ImageNet)")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Final layer: Linear({num_features}, 2)")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Utente/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:10<00:00, 4.53MB/s]


=== Model Loaded ===
Architecture: ResNet18 (pretrained on ImageNet)
Total parameters: 11,177,538
Final layer: Linear(512, 2)


## Freeze Backbone & Configure Training

In [9]:
# Freeze all backbone parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze final layer
for param in model.fc.parameters():
    param.requires_grad = True

# Set BatchNorm to eval mode
def set_bn_eval(m):
    if isinstance(m, nn.BatchNorm2d):
        m.eval()

model.apply(set_bn_eval)

# Verify trainable params
trainable_params = [p for p in model.parameters() if p.requires_grad]
total_trainable = sum(p.numel() for p in trainable_params)

print("=== Training Configuration ===")
print(f"Trainable parameter tensors: {len(trainable_params)}")
print(f"Total trainable parameters: {total_trainable:,}")
print(f"Total model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Backbone frozen")

=== Training Configuration ===
Trainable parameter tensors: 2
Total trainable parameters: 1,026
Total model parameters: 11,177,538
Backbone frozen


## Training Function

In [10]:
def train_and_validate(model, train_loader, val_loader, optimizer, criterion, device, epochs=10):
    """Train and validate the model, return training history"""
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [],  'val_acc': []
    }

    for epoch in range(1, epochs + 1):
        # ---- TRAINING ----
        model.train()
        #Re-freeze BatchNorm after model.train()
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        start = time.time()
        
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X.size(0)
            preds = outputs.argmax(dim=1)
            train_correct += (preds == y).sum().item()
            train_total += y.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # ---- VALIDATION ----
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                loss = criterion(outputs, y)
                val_loss += loss.item() * X.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        elapsed = time.time() - start

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch:2d}/{epochs}  time={elapsed:5.1f}s  "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    return history

## Train the Model

In [11]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
params_to_update = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params_to_update, lr=lr, momentum=momentum, weight_decay=weight_decay)

print("\n=== Starting Training ===")
print(f"Optimizer: SGD(lr={lr}, momentum={momentum})")
print(f"Criterion: CrossEntropyLoss")

# Train
history = train_and_validate(model, train_loader, val_loader, optimizer, criterion, device, epochs=epochs)

print("\n   Training complete!")


=== Starting Training ===
Optimizer: SGD(lr=0.001, momentum=0.9)
Criterion: CrossEntropyLoss
Epoch  1/20  time=  6.6s  train_loss=0.2746 train_acc=0.8182 | val_loss=0.0006 val_acc=1.0000
Epoch  2/20  time=  5.2s  train_loss=0.0002 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch  3/20  time=  5.4s  train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch  4/20  time=  5.1s  train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch  5/20  time=  5.2s  train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch  6/20  time=  5.2s  train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch  7/20  time=  5.1s  train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch  8/20  time=  5.1s  train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch  9/20  time=  5.2s  train_loss=0.0000 train_acc=1.0000 | val_loss=0.0000 val_acc=1.0000
Epoch 10/20  time=  5.0s  train_loss=0.0000 train_acc=1.0000

## Save Trained Model & Training History

In [12]:
# Create models directory
os.makedirs('../models', exist_ok=True)

# Save model weights
model_path = '../models/resnet18_mvtec_binary.pth'
torch.save(model.state_dict(), model_path)
print(f"    Model weights saved to: {model_path}")

# Save training history
history_path = '../models/training_history.pkl'
pickle.dump(history, open(history_path, 'wb'))
print(f"    Training history saved to: {history_path}")

# Save model info
model_info = {
    'architecture': 'ResNet18',
    'pretrained': True,
    'num_classes': 2,
    'class_labels': {0: 'Good', 1: 'Defective'},
    'input_size': 224,
    'normalization': {
        'mean': imagenet_mean,
        'std': imagenet_std
    },
    'training': {
        'epochs': epochs,
        'optimizer': optimizer_type,
        'learning_rate': lr,
        'momentum': momentum,
        'weight_decay': weight_decay
    }
}

info_path = '../models/model_info.json'
with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=2)
print(f"    Model info saved to: {info_path}")

    Model weights saved to: ../models/resnet18_mvtec_binary.pth
    Training history saved to: ../models/training_history.pkl
    Model info saved to: ../models/model_info.json
